 Hinge loss and gradient

In [2]:
def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

In [ ]:
def hinge_loss(X, y, w, b):
    n = len(X)
    total_loss = 0.0
    for i in range(n):
        margin = y[i] * (dot(w, X[i]) + b)
        total_loss += max(0.0, 1.0 - margin)
    return total_loss / n



In [4]:
# Simple test case
X = [[1.0, 2.0],   # sample 1
     [2.0, 1.0],   # sample 2
     [0.5, 0.5]]   # sample 3

y = [1, -1, 1]     # labels (+1 / -1)
w = [1.0, 1.0]     # weight vector
b = 0.0            # bias

print(hinge_loss(X, y, w, b))

1.3333333333333333


Linear SVM via gradient descent

In [6]:
class LinearSVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_epochs=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_epochs = n_epochs
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        n_features = len(X[0])
        self.w = [0.0] * n_features
        self.b = 0.0

        for epoch in range(self.n_epochs):
            for i in range(len(X)):
                margin = y[i] * (dot(self.w, X[i]) + self.b)
                if margin >= 1:
                    self.w = [wj - self.lr * self.lambda_param * wj
                              for wj in self.w]
                else:
                    self.w = [wj - self.lr * (self.lambda_param * wj - y[i] * X[i][j])
                              for j, wj in enumerate(self.w)]
                    self.b -= self.lr * (-y[i])

    def predict(self, X):
        return [1 if dot(self.w, x) + self.b >= 0 else -1 for x in X]

In [7]:
X = [[2.0, 2.0],
     [2.5, 2.5],     # class +1
     [1.5, 2.0],
     [0.0, 0.0],
     [0.5, 0.0],     # class -1
     [-0.5, -0.5]]
y = [1, 1, 1, -1, -1, -1]

model = LinearSVM(lr=0.01, lambda_param=0.01, n_epochs=1000)
model.fit(X, y)

print("w =", model.w)
print("b =", model.b)
print("predictions:", model.predict(X))
print("expected   :", y)

w = [0.36625604197071937, 0.8461027365046957]
b = -1.1900000000000008
predictions: [1, 1, 1, -1, -1, -1]
expected   : [1, 1, 1, -1, -1, -1]


Step 3: Kernel functions

In [8]:
def linear_kernel(x, z):
    return dot(x, z)

def polynomial_kernel(x, z, degree=3, c=1.0):
    return (dot(x, z) + c) ** degree

def rbf_kernel(x, z, gamma=0.5):
    diff = [xi - zi for xi, zi in zip(x, z)]
    return math.exp(-gamma * dot(diff, diff))

In [15]:
import math
x = [1.0, 2.0]
z = [3.0, 4.0]

# ---------- LINEAR KERNEL ----------
# K(x,z) = <x,z> = 1*3 + 2*4 = 11
assert linear_kernel(x, z) == 11.0

# symmetry
assert linear_kernel(x, z) == linear_kernel(z, x)
# self: <x,x> = ||x||^2 = 1+4 = 5
assert linear_kernel(x, x) == 5.0
# orthogonal vectors -> 0
assert linear_kernel([1, 0], [0, 1]) == 0.0
# negative possible (not a distance!)
assert linear_kernel([1, 0], [-1, 0]) == -1.0

# ---------- POLYNOMIAL KERNEL ----------
# K(x,z) = (<x,z> + c)^d, default c=1, d=3
# (<x,z> + 1)^3 = (11 + 1)^3 = 1728
assert polynomial_kernel(x, z) == 12.0 ** 3          # 1728
# custom degree / c: (11 + 2)^2 = 169
assert polynomial_kernel(x, z, degree=2, c=2.0) == 13.0 ** 2
# symmetry
assert polynomial_kernel(x, z) == polynomial_kernel(z, x)
# self: (||x||^2 + c)^d = (5 + 1)^3 = 216
assert polynomial_kernel(x, x) == 6.0 ** 3
# monotonic in degree (for margin > 1 inputs)
assert polynomial_kernel(x, z, degree=4) > polynomial_kernel(x, z, degree=3)

# ---------- RBF KERNEL ----------
# K(x,z) = exp(-gamma * ||x-z||^2)
# ||x-z||^2 = (1-3)^2 + (2-4)^2 = 4 + 4 = 8
# gamma=0.5 -> exp(-0.5*8) = exp(-4)
assert math.isclose(rbf_kernel(x, z), math.exp(-4.0), rel_tol=1e-12)
# self-similarity: K(x,x) = exp(0) = 1  (always exactly 1)
assert rbf_kernel(x, x) == 1.0
assert rbf_kernel(z, z) == 1.0
# symmetry
assert math.isclose(rbf_kernel(x, z), rbf_kernel(z, x), rel_tol=1e-12)
# range: 0 < K <= 1
assert 0.0 < rbf_kernel(x, z) <= 1.0
# farther points -> smaller kernel (distance decay)
near  = rbf_kernel([0.0, 0.0], [0.1, 0.1], gamma=0.5)
far   = rbf_kernel([0.0, 0.0], [10.0, 10.0], gamma=0.5)
assert near > far
# gamma -> 0 : kernel -> 1 ;  gamma large : kernel -> 0 for distinct points
assert math.isclose(rbf_kernel(x, z, gamma=1e-9), 1.0, rel_tol=1e-6)
assert rbf_kernel(x, z, gamma=100.0) < 1e-30

In [16]:
import math
from numpy import dot

def linear_kernel(x, z): return dot(x, z)
def polynomial_kernel(x, z, degree=3, c=1.0): return (dot(x, z) + c) ** degree
def rbf_kernel(x, z, gamma=0.5):
    diff = [xi - zi for xi, zi in zip(x, z)]
    return math.exp(-gamma * dot(diff, diff))

x = [1.0, 2.0]
z = [3.0, 4.0]

tests = []
def check(name, cond):
    tests.append((name, cond))
    print(f"{'PASS' if cond else 'FAIL'}  {name}")

check("linear exact",        linear_kernel(x, z) == 11.0)
check("linear symmetry",     linear_kernel(x, z) == linear_kernel(z, x))
check("linear self",         linear_kernel(x, x) == 5.0)
check("linear orthogonal",   linear_kernel([1, 0], [0, 1]) == 0.0)
check("linear negative",     linear_kernel([1, 0], [-1, 0]) == -1.0)

check("poly exact",          polynomial_kernel(x, z) == 12.0 ** 3)
check("poly custom",         polynomial_kernel(x, z, degree=2, c=2.0) == 13.0 ** 2)
check("poly symmetry",       polynomial_kernel(x, z) == polynomial_kernel(z, x))
check("poly self",           polynomial_kernel(x, x) == 6.0 ** 3)
check("poly monotone",       polynomial_kernel(x, z, degree=4) > polynomial_kernel(x, z, degree=3))

check("rbf exact",           math.isclose(rbf_kernel(x, z), math.exp(-4.0), rel_tol=1e-12))
check("rbf self x",          rbf_kernel(x, x) == 1.0)
check("rbf self z",          rbf_kernel(z, z) == 1.0)
check("rbf symmetry",        math.isclose(rbf_kernel(x, z), rbf_kernel(z, x), rel_tol=1e-12))
check("rbf range",           0.0 < rbf_kernel(x, z) <= 1.0)
check("rbf decay",           rbf_kernel([0.0, 0.0], [0.1, 0.1], gamma=0.5) >
                             rbf_kernel([0.0, 0.0], [10.0, 10.0], gamma=0.5))
check("rbf gamma->0",        math.isclose(rbf_kernel(x, z, gamma=1e-9), 1.0, rel_tol=1e-6))
check("rbf gamma large",     rbf_kernel(x, z, gamma=100.0) < 1e-30)

passed = sum(ok for _, ok in tests)
print(f"\n{passed}/{len(tests)} passed")

PASS  linear exact
PASS  linear symmetry
PASS  linear self
PASS  linear orthogonal
PASS  linear negative
PASS  poly exact
PASS  poly custom
PASS  poly symmetry
PASS  poly self
PASS  poly monotone
PASS  rbf exact
PASS  rbf self x
PASS  rbf self z
PASS  rbf symmetry
PASS  rbf range
PASS  rbf decay
PASS  rbf gamma->0
PASS  rbf gamma large

18/18 passed


Margin and support vector identification

In [17]:
def find_support_vectors(X, y, w, b, tol=1e-3):
    support_vectors = []
    for i in range(len(X)):
        margin = y[i] * (dot(w, X[i]) + b)
        if abs(margin - 1.0) < tol:
            support_vectors.append(i)
    return support_vectors

In [19]:
w = [1.0, 0.0]
b = 0.0

X = [
    [ 1.0,  2.0],   # idx 0: y=+1, margin = +1  -> SV
    [ 2.0,  1.0],   # idx 1: y=+1, margin = +2  -> NOT (outside margin)
    [-1.0,  3.0],   # idx 2: y=-1, margin = +1  -> SV
    [-2.0, -1.0],   # idx 3: y=-1, margin = +2  -> NOT
    [ 1.0, -1.0],   # idx 4: y=+1, margin = +1  -> SV
]
y = [1, 1, -1, -1, 1]

sv = find_support_vectors(X, y, w, b)
print(sv)   # expect [0, 2, 4]
assert sv == [0, 2, 4]

[0, 2, 4]


In [21]:
X = [[ 2.0,  2.0],
              [ 2.5,  2.5],
              [ 1.5,  2.0],
              [ 0.0,  0.0],
              [ 0.5,  0.0],
              [-0.5, -0.5]]
y = [1, 1, 1, -1, -1, -1]

model = LinearSVM(lr=0.01, lambda_param=0.01, n_epochs=1000)
model.fit(X, y)

sv = find_support_vectors(X, y, model.w, model.b, tol=0.05)  # loosen tol for SGD's noisy w

# every reported SV must genuinely be near the margin
for i in sv:
    m = y[i] * (dot(model.w, X[i]) + model.b)
    assert abs(m - 1.0) < 0.05, f"idx {i}: margin {m} not near 1"

print("support vector indices:", sv)
print("their margins:", [round(y[i]*(dot(model.w, X[i])+model.b), 4) for i in sv])

support vector indices: [4]
their margins: [np.float64(1.0069)]
